# Flat MAS for a Toy Travel Agency

This notebook shows a **flat multi-agent architecture** using a simple travel-planning task.
All agents operate at the same level, they all read the same shared board, and nobody acts as a manager.

The learning goal is to see how coordination changes when the system relies on **peer-to-peer updates** instead of top-down delegation.

## What We Will Build

We will use one small travel catalog and one traveler request in all four notebooks.
In this flat version, three peer agents will discuss the same trip:

- one peer leans toward destination and timing,
- one peer leans toward booking choices,
- one peer leans toward activities and overall balance.

The key point is that these are only **specialties**, not ranks. Any peer can challenge or improve any part of the plan.

In [1]:
from typing import Literal

from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

# One small model keeps the examples easy to compare.
# These notebooks assume OPENAI_API_KEY is already available.
model = init_chat_model("openai:gpt-4.1-mini", temperature=0)

USER_REQUEST = """
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan
""".strip()

DESTINATIONS = {
    "Lisbon": {
        "best_period": "April to June",
        "style": "sunny, walkable, relaxed",
        "notes": "great for food, viewpoints, and compact neighborhoods",
    },
    "Barcelona": {
        "best_period": "April to June",
        "style": "lively, artistic, seaside",
        "notes": "strong mix of architecture, beach walks, and tapas",
    },
    "Prague": {
        "best_period": "April to June",
        "style": "historic, compact, lower-cost",
        "notes": "easy sightseeing with a classic old-town atmosphere",
    },
}

FLIGHTS = [
    {
        "destination": "Lisbon",
        "code": "TP-833",
        "price": 180,
        "type": "direct",
        "duration": "3h 05m",
    },
    {
        "destination": "Lisbon",
        "code": "IB-310",
        "price": 150,
        "type": "1 stop",
        "duration": "5h 10m",
    },
    {
        "destination": "Barcelona",
        "code": "VY-611",
        "price": 140,
        "type": "direct",
        "duration": "1h 50m",
    },
    {
        "destination": "Barcelona",
        "code": "IB-220",
        "price": 125,
        "type": "1 stop",
        "duration": "4h 00m",
    },
    {
        "destination": "Prague",
        "code": "FR-721",
        "price": 110,
        "type": "direct",
        "duration": "1h 55m",
    },
    {
        "destination": "Prague",
        "code": "OS-410",
        "price": 145,
        "type": "1 stop",
        "duration": "3h 45m",
    },
]

HOTELS = [
    {
        "destination": "Lisbon",
        "name": "Baixa Stay",
        "price_per_night": 145,
        "style": "central boutique hotel",
    },
    {
        "destination": "Lisbon",
        "name": "River Rooms",
        "price_per_night": 120,
        "style": "simple hotel near transit",
    },
    {
        "destination": "Barcelona",
        "name": "Born Hotel",
        "price_per_night": 160,
        "style": "central design hotel",
    },
    {
        "destination": "Barcelona",
        "name": "Gracia Inn",
        "price_per_night": 130,
        "style": "quiet hotel in a walkable district",
    },
    {
        "destination": "Prague",
        "name": "Old Town House",
        "price_per_night": 115,
        "style": "historic hotel near main sights",
    },
    {
        "destination": "Prague",
        "name": "City Garden Hotel",
        "price_per_night": 95,
        "style": "budget-friendly hotel with tram access",
    },
]

ACTIVITIES = [
    {
        "destination": "Lisbon",
        "name": "Alfama food walk",
        "tag": "food",
        "price": 35,
    },
    {
        "destination": "Lisbon",
        "name": "Belem and river tram day",
        "tag": "culture",
        "price": 25,
    },
    {
        "destination": "Barcelona",
        "name": "Gothic Quarter tapas evening",
        "tag": "food",
        "price": 40,
    },
    {
        "destination": "Barcelona",
        "name": "Sagrada Familia and modernism route",
        "tag": "culture",
        "price": 32,
    },
    {
        "destination": "Prague",
        "name": "Old Town walking tour",
        "tag": "culture",
        "price": 18,
    },
    {
        "destination": "Prague",
        "name": "Czech dinner and jazz night",
        "tag": "food",
        "price": 30,
    },
]


def flights_for(destination: str) -> list[dict]:
    return [flight for flight in FLIGHTS if flight["destination"] == destination]



def hotels_for(destination: str) -> list[dict]:
    return [hotel for hotel in HOTELS if hotel["destination"] == destination]



def activities_for(destination: str) -> list[dict]:
    return [activity for activity in ACTIVITIES if activity["destination"] == destination]



def catalog_text() -> str:
    lines = []

    for destination, info in DESTINATIONS.items():
        lines.append(f"Destination: {destination}")
        lines.append(f"  Best period: {info['best_period']}")
        lines.append(f"  Style: {info['style']}")
        lines.append(f"  Notes: {info['notes']}")
        lines.append("  Flights:")

        for flight in flights_for(destination):
            lines.append(
                f"    - {flight['code']} | {flight['type']} | EUR {flight['price']} | {flight['duration']}"
            )

        lines.append("  Hotels:")
        for hotel in hotels_for(destination):
            lines.append(
                f"    - {hotel['name']} | EUR {hotel['price_per_night']} per night | {hotel['style']}"
            )

        lines.append("  Activities:")
        for activity in activities_for(destination):
            lines.append(
                f"    - {activity['name']} | {activity['tag']} | EUR {activity['price']}"
            )

        lines.append("")

    return "\n".join(lines).strip()


CATALOG_TEXT = catalog_text()
print(USER_REQUEST)

Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan


## Define the Peer Agents

We use a tiny structured output schema so every peer produces the same shape:

1. one update for the shared board,
2. one stop vote.

This keeps the code minimal and makes the orchestration easy to read.

In [2]:
class PeerTurn(BaseModel):
    update: str = Field(description="One short update to the shared travel board.")
    ready: bool = Field(description="True when this peer thinks the group can stop.")


peer_llm = model.with_structured_output(PeerTurn)

PEERS = {
    "Destination peer": (
        "Focus on destination choice and the best travel period, "
        "but you may challenge any earlier choice."
    ),
    "Booking peer": (
        "Focus on flight and hotel fit, "
        "but you may challenge any earlier choice."
    ),
    "Experience peer": (
        "Focus on activities and trip balance, "
        "but you may challenge any earlier choice."
    ),
}


def run_peer(peer_name: str, specialty: str, shared_board: list[str]) -> PeerTurn:
    # Every peer sees the same board.
    # That is the key property of the flat architecture in this notebook.
    board_text = "\n".join(shared_board)

    messages = [
        SystemMessage(
            content=(
                "You are one peer in a flat travel-agency MAS. "
                "There is no manager. You can refine, disagree with, or replace earlier ideas. "
                "Only use destinations, flights, hotels, and activities from the catalog.\n\n"
                f"Your specialty: {specialty}"
            )
        ),
        HumanMessage(
            content=(
                f"Traveler request:\n{USER_REQUEST}\n\n"
                f"Catalog:\n{CATALOG_TEXT}\n\n"
                f"Shared board so far:\n{board_text}\n\n"
                "Add one useful update to the plan and say whether the group can stop."
            )
        ),
    ]

    return peer_llm.invoke(messages)

## Run the Flat Conversation

The loop below is deliberately small.
We do not use a graph because a basic Python loop is enough for this architecture.

Notice the orchestration rule:
every agent reads the **same** board and writes back to the **same** board.

In [3]:
shared_board = [
    "Start a travel plan that satisfies the traveler request.",
]

# We keep the loop very small on purpose.
# The point is to show peer-to-peer coordination, not build a complex runtime.
for round_number in range(2):
    ready_votes = []

    for peer_name, specialty in PEERS.items():
        turn = run_peer(peer_name, specialty, shared_board)
        shared_board.append(f"Round {round_number + 1} | {peer_name}: {turn.update}")
        ready_votes.append(turn.ready)

    # In a flat system, the peers decide when the conversation can stop.
    if all(ready_votes):
        break

shared_board_text = "\n".join(shared_board)

final_itinerary = model.invoke(
    [
        SystemMessage(
            content=(
                "Turn the shared travel board into one clean final itinerary. "
                "This is only a formatting step, not a manager role."
            )
        ),
        HumanMessage(
            content=(
                f"Traveler request:\n{USER_REQUEST}\n\n"
                f"Shared board:\n{shared_board_text}"
            )
        ),
    ]
).content

print("=== Shared board ===")
for item in shared_board:
    print("-", item)

print("\n=== Final itinerary ===")
print(final_itinerary)

=== Shared board ===
- Start a travel plan that satisfies the traveler request.
- Round 1 | Destination peer: Suggest Lisbon as the destination for the 4-day spring trip from Rome. It fits the mid-range budget with direct flights (TP-833 at EUR 180) and central hotels like Baixa Stay (EUR 145 per night). Lisbon offers a relaxed, walkable city with great food and cultural activities such as the Alfama food walk and Belem river tram day, perfect for a simple daily plan.
- Round 1 | Booking peer: Lisbon is a great choice with direct flights and a central boutique hotel option. The Alfama food walk and Belem river tram day provide a balanced mix of food and culture for the 4-day trip.
- Round 1 | Experience peer: I suggest choosing the Baixa Stay hotel for a central and comfortable stay, and planning the Alfama food walk on day 1 and the Belem and river tram day on day 2 to balance food and culture activities early in the trip.
- Round 2 | Destination peer: Confirm Lisbon as the destinatio

## Why This Fits the Flat Architecture

- There is no manager.
- Every peer can react directly to the same shared state.
- The stop condition comes from peer agreement, not a top-level controller.

That is the core behavior we want beginners to see in a flat MAS.